# 03 モデル実行 (予約 & 天気)
元ノートの walkforward 実行部を再構成。必要なライブラリは scripts ディレクトリから import。

In [ ]:
import pandas as pd, sys, importlib
from datetime import timedelta
from sklearn.metrics import r2_score, mean_absolute_error
sys.path.insert(0,'/works/scripts')
from new_model1 import full_walkforward as fw1, ReserveFeatureBuilder as RB1
from new_model2.predict_model_v4_2_4 import full_walkforward as fw2
from new_model2.feature_builder import WeatherFeatureBuilder, ReserveFeatureBuilder as RB2
print('[INFO] imports ok')

In [ ]:
# 前段ノート(01)で保存した中間CSVをここで読む想定 (軽量化のため)。
# 直接再読み込みも可。
df_all = pd.read_csv('/works/data/2023_all.csv', encoding='utf-8')  # placeholder: 実際は統合済みを保存して利用推奨
# ... 実際の再現には 01 の処理結果を parquet などで保存しここで読み直す設計に。
print('[WARN] 簡略化: df_all を単一CSVから読み込み (本番は01の成果を使用)')

In [ ]:
# 予約特徴量モデル (new_model1) の例示的実行雛形
def run_reserve_model(df_all, df_reserve, days=365):
    df_reserve_feat = RB1(df_reserve).build()
    reserve_dates = df_reserve_feat.index
    df_all['伝票日付'] = pd.to_datetime(df_all['伝票日付'])
    df_all_f = df_all[df_all['伝票日付'].isin(reserve_dates)].copy()
    latest_date = df_all_f['伝票日付'].max()
    cutoff = latest_date - pd.Timedelta(days=days)
    subset = df_all_f[df_all_f['伝票日付'] >= cutoff].copy()
    hol_min, hol_max = subset['伝票日付'].min(), subset['伝票日付'].max()
    from datetime import date
    holidays = []  # 省略: 祝日計算は本来 get_japanese_holidays を再利用
    actual, pred, model = fw1(subset, holidays=holidays, df_reserve=df_reserve, min_stage1_days=20, min_stage2_days=10, top_n=2, return_model=True)
    if isinstance(actual, list) and len(actual)>0:
        r2 = r2_score(actual, pred); mae = mean_absolute_error(actual, pred)
        print(f'[RESULT reserve] R2={r2:.3f} MAE={mae:.1f}')
    return model
print('関数定義完了')

In [ ]:
# 天気モデル (new_model2) 雛形
def run_weather_model(df_all, df_reserve, days=365):
    df_all['伝票日付'] = pd.to_datetime(df_all['伝票日付'])
    latest = df_all['伝票日付'].max(); cutoff = latest - pd.Timedelta(days=days)
    subset = df_all[df_all['伝票日付'] >= cutoff].copy()
    hol_min, hol_max = subset['伝票日付'].min(), subset['伝票日付'].max()
    w_builder = WeatherFeatureBuilder(start_date=hol_min, end_date=hol_max, enable_fallback=True)
    df_weather_full = w_builder.build()
    df_weather = df_weather_full.loc[hol_min:hol_max] if not df_weather_full.empty else df_weather_full
    actual, pred, model = fw2(df_raw=subset, df_reserve=df_reserve, holidays=[], df_weather=df_weather, min_stage1_days=30, min_stage2_days=15, top_n=2, return_model=True)
    if isinstance(actual, list) and len(actual)>0:
        r2 = r2_score(actual, pred); mae = mean_absolute_error(actual, pred)
        print(f'[RESULT weather] R2={r2:.3f} MAE={mae:.1f}')
    return model
print('関数定義完了')

詳細な結果・可視化は 04 へ分離。